In [ ]:
import os
os.chdir('???')
os.getcwd()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

### Load Dataset

In [ ]:
orig_df = pd.read_csv("wholesale_customers_data.csv")
orig_df.head(10)

In [ ]:
orig_df.hist()

In [ ]:
df = orig_df.copy()

### Handling missing values

In [ ]:
# Plot heatmap to visualize locations of missing values
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')

### Handling outliers

In [ ]:
def remove_outliers_iqr(df, outlier_vars):
    temp_df = df.copy()
    for _var in outlier_vars:
        q1 = temp_df[_var].quantile(0.25)
        q3 = temp_df[_var].quantile(0.75)
        iqr = q3-q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        mask = (temp_df[_var] < lower_bound) | (temp_df[_var] > upper_bound)
        temp_df = temp_df[~mask]
        print(temp_df.shape)
    col_names = df.columns.tolist()
    new_df = pd.DataFrame(temp_df, col_names)
    return temp_df

### Transform skewed data

In [ ]:
def transform_log(X, skew_var):
    temp_X = X.copy()
    temp_X[skew_var] = np.log1p(temp_X[skew_var])   # log1p = log(1 + y)
    return temp_X

def transform_boxcox(X, skew_var):
    temp_X = X.copy()
    from scipy.stats import boxcox
    temp_bc, lambda_bc = boxcox(temp_X[skew_var])
    temp_X[skew_var] = temp_bc
    return temp_X

def transform_yeojohnson(X, skew_var):
    temp_X = X.copy()
    from sklearn.preprocessing import PowerTransformer
    pt = PowerTransformer(method='yeo-johnson')
    temp_X[skew_var] = pt.fit_transform(temp_X[[skew_var]])
    return temp_X

### Encoding categorical data

In [ ]:
# One-Hot Encoder version 1: convert all categorical variables
def onehot_encoder_all(df):
    from sklearn.preprocessing import OneHotEncoder
    onehot_encoder = OneHotEncoder(sparse_output=False,        # use sparse=True for large data
                                    handle_unknown='ignore')
    oh_encoded = onehot_encoder.fit_transform(df)
    ohe_df = pd.DataFrame(oh_encoded,
                         columns=onehot_encoder.get_feature_names_out(df.columns))
    return ohe_df

# One-Hot Encoder version 2: Convert selected variables
def onehot_encoder_selected(df, encoded_vars):
    df = pd.get_dummies(df, columns=encoded_vars, drop_first=False)
    return df

# Label Encoder
def label_encoder(df, encoded_var):
    from sklearn.preprocessing import LabelEncoder
    label_encoder = LabelEncoder()
    df[encoded_var] = label_encoder.fit_transform(df[encoded_var])
    return df

# Ordinal Encoder
def ordinal_encoder(df, encoded_var, encode_order):
    from sklearn.preprocessing import OrdinalEncoder
    ordinal_encoder = OrdinalEncoder(categories=[encode_order])
    df[encoded_var] = ordinal_encoder.fit_transform(df[[encoded_var]])
    return df

In [ ]:
# Frequenct Encoder
def frequent_encoder(df, encoded_var):
    freq_encoding = df[encoded_var].value_counts()
    df[encoded_var] = df[encoded_var].map(freq_encoding)
    return df

# Target Encoder
def target_encoder(df, encoded_var, encoded_num):
    import category_encoders as ce
    target_encoder = ce.TargetEncoder(cols=[encoded_var])
    df[encoded_var] = target_encoder.fit_transform(df[encoded_var], df[encoded_num])
    return df

# Binary Encoder
def binary_encoder(df, encoded_var):
    import category_encoders as ce
    binary_encoder = ce.BinaryEncoder(cols=[encoded_var])
    df_binary = binary_encoder.fit_transform(df[encoded_var])
    df_binary.columns = [f"{encoded_var}_bin_{i}" for i in range(df_binary.shape[1])]
    df = pd.concat([df, df_binary], axis=1)
    df = df.drop([encoded_var], axis=1)
    return df

### Hash Encoder
def hash_encoder(df, encoded_var, hash_len):
    import category_encoders as ce
    hash_encoder = ce.HashingEncoder(cols=[encoded_var], n_components=hash_len)  # n_components = length of hash, default = 8
    df_hash = hash_encoder.fit_transform(df[encoded_var])
    df_hash.columns = [f"{encoded_var}_hash_{i}" for i in range(df_hash.shape[1])]
    df = pd.concat([df, df_hash], axis=1)
    df = df.drop([encoded_var], axis=1)
    return df

### Scaling

In [ ]:
# Scale by standardized normal distribution, (x-mean)/sd
def scale_standard(X):
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled

# Scale by min-max, (x-min)/(max-min)
def scale_minmax(X):
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled

# Scale by robust, (x-median)/iqr
def scale_minmax(X):
    from sklearn.preprocessing import RobustScaler
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled

### Removing Redundancy

In [ ]:
def remove_highly_correlated(df, threshold=0.9):
    corr_matrix = df.corr().abs()

    # Upper triangle of correlation matrix
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Identify features to drop
    dropped_vars = [column for column in upper.columns if any(upper[column] > threshold)]

    # Drop redundant features
    df_reduced = df.drop(columns=dropped_vars)

    return df_reduced, dropped_vars

def remove_low_variance(df, threshold=0.01):
    from sklearn.feature_selection import VarianceThreshold

    selector = VarianceThreshold(threshold)
    df_reduced = selector.fit_transform(df)

    kept_vars = df.columns[selector.get_support()]
    kept_vars = kept_vars.tolist()

    new_df = pd.DataFrame(df_reduced, columns=kept_vars)
    return new_df, kept_vars

### Performance Evaluation

In [ ]:
def dunn_index(X, labels):
    unique_clusters = np.unique(labels)

    # Compute cluster centroids
    centroids = np.array([
        X[labels == k].mean(axis=0) for k in unique_clusters
    ])

    from scipy.spatial.distance import cdist
    # Inter-cluster distance (minimum distance between clusters)
    inter_cluster_dist = np.min(
        cdist(centroids, centroids)[np.triu_indices(len(unique_clusters), k=1)]
    )

    # Intra-cluster distance (maximum cluster diameter)
    intra_cluster_dist = max(
        np.max(cdist(X[labels == k], X[labels == k]))
        for k in unique_clusters
    )
    return inter_cluster_dist / intra_cluster_dist

def evaluate_clustering(X, X_clusters):
    from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
    silhouette = silhouette_score(X, X_clusters)
    db = davies_bouldin_score(X, X_clusters)
    dunn = dunn_index(X, X_clusters)
    ch = calinski_harabasz_score(X, X_clusters)
    return silhouette, db, dunn, ch

### Interpretability

In [ ]:
def create_radar_graph(X, X_clusters):
    clustered_df = X.copy()
    clustered_df["Cluster"] = X_clusters
    cluster_means = clustered_df.groupby("Cluster").mean()

    # plot Radar Chart  for cluster means
    labels = cluster_means.columns
    num_features = len(labels)

    angles = np.linspace(0, 2 * np.pi, num_features, endpoint=False).tolist()
    angles += angles[:1]

    plt.figure(figsize=(6, 6))

    for i, row in cluster_means.iterrows():
        values = row.tolist()
        values += values[:1]
        plt.polar(angles, values, label=f"Cluster {i}")

    plt.xticks(angles[:-1], labels, fontsize=9)
    plt.title("Cluster Profiles (Feature Means)")
    plt.legend()
    plt.show()

In [ ]:
def create_feature_boxplots(X, X_clusters):
    clustered_df = X.copy()
    clustered_df["Cluster"] = X_clusters
    cluster_means = clustered_df.groupby("Cluster").mean()

    # plot boxplots of feature values
    for col in X.columns:
        clustered_df.boxplot(column=col, by="Cluster", figsize=(6, 4))
        plt.title(f"{col} by Cluster")
        plt.suptitle("")
        plt.show()

In [ ]:
def create_pca_scatterplot(X_scaled, X_clusters):
    from sklearn.decomposition import PCA

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)

    plt.figure(figsize=(7,5))
    unique_clusters = np.unique(X_clusters)
    for cluster_id in unique_clusters:
        idx = X_clusters == cluster_id
        plt.scatter(X_pca[idx, 0],
                    X_pca[idx, 1],
                    s=40,
                    label=f"Cluster {cluster_id}"
                   )

    plt.title("PCA 2D Scatterplot of Clusters")
    plt.xlabel("PC 1")
    plt.ylabel("PC 2")
    plt.legend(title="Cluster ID")
    plt.show()

In [ ]:
def get_permutation_importance(X_scaled, X_clusters):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.inspection import permutation_importance

    # Construct proxy classifier
    rf = RandomForestClassifier(random_state=23)
    rf.fit(X_scaled, X_clusters)

    # Compute permutation importance
    perm_importance = permutation_importance(rf, X_scaled, X_clusters, n_repeats=10, random_state=23)
    importance_df = pd.DataFrame({
        'Feature': df.columns,
        'Importance': perm_importance['importances_mean']
    }).sort_values(by='Importance', ascending=False)

    return importance_df

In [ ]:
def create_shap_values_plot(X_scaled, X_clusters, selected_clusterID=0):
    import shap
    from sklearn.ensemble import RandomForestClassifier

    # Construct proxy classifier
    rf = RandomForestClassifier(random_state=23)
    rf.fit(X_scaled, X_clusters)

    # Use TreeExplainer for RandomForest
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_scaled)

    # Visualize feature importance for selected cluster ID
    shap.summary_plot(shap_values[:,:,selected_clusterID], X_scaled, feature_names=df.columns)

### Experiments

#### Prepare data

In [ ]:
df = orig_df.copy()
print(df.columns)
df.head()

In [ ]:
df = df.drop(['Channel', 'Region'], axis=1)

#### Data Preprocssing

In [ ]:
df.hist()

In [ ]:
# Remove outliers
outlier_vars = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
df = remove_outliers_iqr(df, outlier_vars)

In [ ]:
df.hist()

In [ ]:
# Correlation-based filtering
df, dropped_vars = remove_highly_correlated(df)
print(dropped_vars)
df.shape

In [ ]:
# Scaling
X = df.copy()
X_scaled = scale_standard(X)
X_scaled

In [ ]:
# Low-variance filtering
df, kept_vars = remove_low_variance(df)
print(kept_vars)
df.shape

### K-Mean Clustering Modeling and Performance Evaluation

In [ ]:
#######################################################
# Construct a kmean clustering model
from sklearn.cluster import KMeans
selected_k = 5  # number of resulting clusters
selected_n_init = 10    # number of times the k-means algorithm is run with different centroid seeds
model = KMeans(n_clusters=selected_k, n_init=selected_n_init, random_state=23)

X_clusters = model.fit_predict(X_scaled)
#######################################################

# Performance Evaluation
silhouette, db, dunn, ch = evaluate_clustering(X_scaled, X_clusters)
print('Silhouette Index: ', silhouette)
print('Davies-Bouldin Index: ', db)
print('Dunn Index: ', dunn)
print('CH Index: ', ch)

In [ ]:
# Construct elbow curve and silhoulette curve to help choose value of k

from sklearn.metrics import silhouette_score

wcss = []
silhouette_scores = []

k_range = range(2, 11)  # silhouette is undefined for k=1

for k in k_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=23)
    temp_X_clusters = kmeans.fit_predict(X_scaled)

    # Elbow (WCSS)
    wcss.append(kmeans.inertia_)

    # Silhouette
    silhouette = silhouette_score(X_scaled, temp_X_clusters)
    silhouette_scores.append(silhouette)

fig, axes = plt.subplots(2, 1, figsize=(8, 8))

axes[0].plot(k_range, wcss, marker='o')
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("WCSS")
axes[0].set_title("Elbow Method")
axes[0].grid(True)

axes[1].plot(k_range, silhouette_scores, marker='o')
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].grid(True)
#axes[1].set_title("Silhouette Score vs Number of Clusters")
plt.show()

### Hierarchical Clustering and Performance Evaluation

In [ ]:
#######################################################
# Construct a hierarchical model

from sklearn.cluster import AgglomerativeClustering

# Set up parameter values
selected_linkage = 'ward'  # 'others are 'complete', 'single', 'average', 'ward'
selected_distance_threshold = 0   # Set to be zero to plot dendrogram
selected_n_clusters = None     # if distance_threshold is 0, this must be None

model=AgglomerativeClustering(linkage=selected_linkage,
                              n_clusters=selected_n_clusters,
                              distance_threshold=selected_distance_threshold)
X_clusters = model.fit(X_scaled)
#######################################################

In [ ]:
# Function to plot dendrogram of hierarchical clustering result
def plot_dendrogram(model, **kwargs):
    # Create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack([model.children_, model.distances_,
                                      counts]).astype(float)

    # Plot the corresponding dendrogram
    from scipy.cluster.hierarchy import dendrogram
    dendrogram(linkage_matrix, **kwargs)

In [ ]:
# Plot dendrogram of hierarchical clustering result
plot_dendrogram(model, truncate_mode='level')

In [ ]:
# Partition dendrogram into clusters

selected_linkage = 'ward'  # 'others are 'complete', 'single', 'average', 'ward'
# To partition dendrogram to clusters, either choose to do 1) or 2) [but not both]
# 1) Set distance_threshold to be cut-off distance (looking for cut-off distance on y-axis of dendrogram)
#    If distance_threshold is not None, set n_clusters to be None.
selected_distance_threshold = 15
selected_n_clusters = None
# 2) Set n_clusters to be your selected number of clusters
#    If n_clusters is not None, set distance_threshold to be None.
#selected_distance_threshold = None
#selected_n_clusters = 5

model=AgglomerativeClustering(linkage=selected_linkage,
                              n_clusters=selected_n_clusters,
                              distance_threshold=selected_distance_threshold)

X_clusters = model.fit_predict(X_scaled)

# Performance Evaluation
silhouette, db, dunn, ch = evaluate_clustering(X_scaled, X_clusters)
print('Silhouette Index: ', silhouette)
print('Davies-Bouldin Index: ', db)
print('Dunn Index: ', dunn)
print('CH Index: ', ch)
X_clusters

### DBSCAN Clustering and Performance Evaluation

In [ ]:
# Find out appropriate distance for k-nn neighbor
# k and distance at elbow will help choose eps and min_samples

from sklearn.neighbors import NearestNeighbors
selected_k = 3
neighbors = NearestNeighbors(n_neighbors=selected_k)
neighbor_model = neighbors.fit(X_scaled)
distances, indices = neighbor_model.kneighbors(X_scaled)
distances = np.sort(distances, axis=0)
plt.plot(distances[:,1])
plt.xlabel('Points sorted based on distance of '+str(selected_n)+' neighbors')
plt.ylabel(str(selected_n)+'-nearest neighbor distance')
plt.show()

In [ ]:
#######################################################
# Construct a model

# Choose value for eps and min_samples
selected_eps = 1
selected_min_samples = 3

from sklearn.cluster import DBSCAN
model=DBSCAN(eps=selected_eps,min_samples=selected_min_samples)
X_clusters=model.fit_predict(X_scaled)

#######################################################

# Performance Evaluation
silhouette, db, dunn, ch = evaluate_clustering(X_scaled, X_clusters)
print('Silhouette Index: ', silhouette)
print('Davies-Bouldin Index: ', db)
print('Dunn Index: ', dunn)
print('CH Index: ', ch)

In [ ]:
X_clusters

In [ ]:
pd.Series(X_clusters).unique()